# CODY-SAM3 — Tier 3 CV at `n_estimators=8` (v3 clean)

**Goal**: run Tier 3 5-fold + LOPO 25-fold at `n_estimators=8` (same conditions as our Tier 2 LOPO, for a rigorous comparison).

**Why n_estimators=8 rather than 24**: on Colab Pro+ with 95 GB VRAM + 176 GB RAM, n_est=24 on Tier 3 (6346 features) peaks at 74 GB VRAM + 126 GB RAM and crashes. n_est=8 divides the buffer by 3 (~25 GB VRAM / 40 GB RAM) and gives a fair comparison with Tier 2 LOPO, which was already run at n_est=8.

**Workflow**:
1. Setup (GPU + Drive + dependencies)
2. Anti-idle
3. Tier 3 5-fold n_est=8 (written directly to Drive)
4. Tier 3 LOPO 25-fold n_est=8 (written directly to Drive)
5. Metrics + calibration
6. Final recap

**Persistence**: every result is written **directly** to Drive. If Colab disconnects, re-running the same cell resumes where it stopped (incremental save after each label).

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

# === Paths ===
DRIVE_SRC  = '/content/drive/MyDrive/sam_3_colab'
DRIVE_OUT  = '/content/drive/MyDrive/sam_3_colab_results'
WORK       = '/content/sam_3_colab'

FOLD5_DIR = os.path.join(DRIVE_OUT, 'tier3_cv_5fold_nest8')
LOPO_DIR  = os.path.join(DRIVE_OUT, 'tier3_cv_25fold_nest8')

assert os.path.isdir(DRIVE_SRC), f'Folder not found: {DRIVE_SRC}'
print(f'Source Drive : {DRIVE_SRC}')
print(f'Output Drive : {DRIVE_OUT}')

# Reset le working dir local
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)

# Copy the pipeline (fast, it is small)
print('\nCopy cody_sam3_pipeline...')
shutil.copytree(os.path.join(DRIVE_SRC, 'cody_sam3_pipeline'),
                os.path.join(WORK, 'cody_sam3_pipeline'))

# Copy the Tier 3 bundle
src_t3 = os.path.join(DRIVE_SRC, 'runs/sam3_tier3')
dst_t3 = os.path.join(WORK, 'runs/sam3_tier3')
os.makedirs(dst_t3)
for fn in ['bundle_meta.json',
           'features_windows_train__t3__fps30__rn1.csv.gz',
           'training_label_summary.csv']:
    src_f = os.path.join(src_t3, fn)
    if os.path.exists(src_f):
        size_mb = os.path.getsize(src_f) / 1024**2
        print(f'Copy {fn} ({size_mb:.1f} MB)...')
        shutil.copy2(src_f, os.path.join(dst_t3, fn))

os.makedirs(DRIVE_OUT, exist_ok=True)

print('\nSetup pret.')

In [ ]:
!pip install -q tabicl antropy

import torch, tabicl, antropy
print(f'torch  : {torch.__version__}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Anti-idle (start BEFORE the long CV runs)

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function ClickConnect(){
  console.log("Anti-idle ping");
  const colab = document.querySelector("#top-toolbar > colab-connect-button");
  if (colab && colab.shadowRoot) {
    const btn = colab.shadowRoot.querySelector("#connect");
    if (btn) btn.click();
  }
}
setInterval(ClickConnect, 60000);
console.log("Anti-idle activated");
'''))
print('Anti-idle JS injecte.')

In [ ]:
%cd /content/sam_3_colab

## 3. Tier 3 5-fold CV at `n_estimators=8`

**Estimated time**: 15-45 min.

Written directly to Drive (`tier3_cv_5fold_nest8`). Saved after each label.

In [ ]:
!python -u cody_sam3_pipeline/cv_sam3.py \
    --bundle_dir runs/sam3_tier3 \
    --out_dir /content/drive/MyDrive/sam_3_colab_results/tier3_cv_5fold_nest8 \
    --n_splits 5 \
    --n_estimators 8 \
    --predict_batch_size 64

### Check: 5-fold progress

In [ ]:
import pandas as pd
from pathlib import Path

p = Path(FOLD5_DIR) / 'cv_oof_patient_predictions.csv'
if p.exists():
    df = pd.read_csv(p)
    print(f'Labels completed     : {sorted(df.label.unique())}')
    print(f'Total lignes patient: {len(df)}')
    print(df.groupby('label').size().to_string())
else:
    print(f'Not started yet. Expected file: {p}')

## 4. Tier 3 LOPO 25-fold at `n_estimators=8`

**Estimated time**: 2-6 h. Incremental saves, written directly to Drive.

**Before running**: confirm with the check cell above that the 5-fold run is complete (8 labels).

In [ ]:
!python -u cody_sam3_pipeline/cv_sam3.py \
    --bundle_dir runs/sam3_tier3 \
    --out_dir /content/drive/MyDrive/sam_3_colab_results/tier3_cv_25fold_nest8 \
    --n_splits 25 \
    --n_estimators 8 \
    --predict_batch_size 64

### Check: LOPO progress

In [ ]:
p = Path(LOPO_DIR) / 'cv_oof_patient_predictions.csv'
if p.exists():
    df = pd.read_csv(p)
    print(f'Labels completed     : {sorted(df.label.unique())}')
    print(f'Total lignes patient: {len(df)}')
    print(df.groupby('label').size().to_string())
else:
    print(f'Not started yet. Expected file: {p}')

## 5. Pooled metrics + threshold calibration

Run **once both CV runs are finished**.

In [ ]:
# 5-fold
!python cody_sam3_pipeline/cv_aggregate_oof.py --cv_dir {FOLD5_DIR}
!python cody_sam3_pipeline/tune_threshold.py --cv_dir {FOLD5_DIR} --level patient --objective f1
!python cody_sam3_pipeline/cv_aggregate_oof.py --cv_dir {FOLD5_DIR} --thresholds {FOLD5_DIR}/thresholds_patient_f1.json

In [ ]:
# LOPO
!python cody_sam3_pipeline/cv_aggregate_oof.py --cv_dir {LOPO_DIR}
!python cody_sam3_pipeline/tune_threshold.py --cv_dir {LOPO_DIR} --level patient --objective f1
!python cody_sam3_pipeline/cv_aggregate_oof.py --cv_dir {LOPO_DIR} --thresholds {LOPO_DIR}/thresholds_patient_f1.json

## 6. Final recap

In [ ]:
for sub_dir, label in [(FOLD5_DIR, '5-fold n_est=8'), (LOPO_DIR, 'LOPO n_est=8')]:
    if not os.path.isdir(sub_dir):
        print(f'{label}: MISSING'); continue
    print(f'\n=== {label} ===  ({sub_dir})')
    for f in sorted(os.listdir(sub_dir)):
        full = os.path.join(sub_dir, f)
        size_kb = os.path.getsize(full) / 1024
        print(f'  {f:<45s} {size_kb:8.1f} KB')

## Done

On your PC, download from Drive:
- `tier3_cv_5fold_nest8/` -> `sam_3/runs/sam3_tier3/cv_5fold_nest8/`
- `tier3_cv_25fold_nest8/` -> `sam_3/runs/sam3_tier3/cv_25fold/`

Final comparison table:

| Tier | 5-fold | LOPO 25-fold |
|---|---|---|
| Tier 1 | n_est=24 (PC) | n_est=24 (PC) |
| Tier 2 | n_est=4 (PC) | n_est=8 (PC) |
| Tier 3 | **n_est=8 (Colab)** + n_est=2 (PC) | **n_est=8 (Colab)** |

Tier 2 LOPO and Tier 3 LOPO are now run under **identical** conditions (n_est=8) -> rigorous comparison for the paper.